## Data Validation

This notebook performs rule-based validation on cleaned expense data.
It ensures the dataset is structurally correct, complete, and compliant
with business rules before KPI computation.

Inputs:
- Cleaned expense dataset

Outputs:
- Validated dataframe
- Validation summary report


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:, .2f}".format)

In [1]:
import os

print("Working directory:", os.getcwd())
print("\nParquet files found:\n")

for root, dirs, files in os.walk("../data"):
    for file in files:
        if file.endswith(".parquet"):
            print(os.path.join(root, file))


Working directory: /Users/vivekduggal/Documents/Projects/orionx-travel-expense-kpis/notebooks

Parquet files found:

../data/feature/expense_features.parquet
../data/processed/expense_cleaned.parquet
../data/curated/expense_kpis.parquet


In [3]:
import pandas as pd
INPUT_PATH = "../data/processed/expense_cleaned.parquet"

import os
assert os.path.exists(INPUT_PATH), "❌ File not found"

df = pd.read_parquet(INPUT_PATH)
df.shape, df.head()


((70000, 31),
              employee  employee_id active            department_name  \
 0  Marvin Bonilla DDS       526243     No                      Legal   
 1        Kenneth Boyd       665075    Yes                        R&D   
 2         John Murray       106822    Yes  Global Support & Services   
 3        Kristi Jones       523319    Yes                  Workplace   
 4  Felicia Tucker DDS       356525    Yes                     People   
 
               job_family           payment_type  \
 0   Software Engineering   Personal Credit Card   
 1  Solutions Engineering           Company Paid   
 2   Software Engineering   Personal Credit Card   
 3   Software Engineering           Company Paid   
 4   Software Engineering  Corporate Credit Card   
 
                  approval_status             report_name  \
 0  Approved in Accounting Review    Third Expense Report   
 1          Sent Back to Employee    Whose Expense Report   
 2                       Approved    Three Expens

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 31 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   employee                          70000 non-null  object        
 1   employee_id                       70000 non-null  int64         
 2   active                            70000 non-null  object        
 3   department_name                   70000 non-null  object        
 4   job_family                        70000 non-null  object        
 5   payment_type                      70000 non-null  object        
 6   approval_status                   70000 non-null  object        
 7   report_name                       70000 non-null  object        
 8   report_id                         70000 non-null  object        
 9   parent_expense_type               70000 non-null  object        
 10  expense_type                      70000 non-nu

In [5]:
# Column Presence Guard (Must-Have)
EXPECTED_COLUMNS = {
    "employee",
    "employee_id",
    "active",
    "department_name",
    "job_family",
    "payment_type",
    "approval_status",
    "report_id",
    "parent_expense_type",
    "expense_type",
    "transaction_date",
    "transaction_year",
    "transaction_month",
    "expense_approved_amount",
    "expense_approved_amount_rpt",
    "reporting_currency",
    "submission_delay_days"
}

missing = EXPECTED_COLUMNS - set(df.columns)
extra = set(df.columns) - EXPECTED_COLUMNS

missing, extra

(set(),
 {'accounting_approval_date',
  'country',
  'difference_b_w_submission_transc',
  'first_submitted_date',
  'location',
  'manager_approval_date',
  'number_of_attendees',
  'purpose',
  'region',
  'reimbursement_currency',
  'report_name',
  'subsidiary_name',
  'transaction_quarter',
  'vendor'})

In [6]:
# MANDATORY NULL CHECKS (HARD FAIL)

MANDATORY_COLUMNS = [
    "employee_id",
    "transaction_date",
    "expense_approved_amount",
    "reporting_currency",
    "approval_status"
]

df[MANDATORY_COLUMNS].isnull().sum()

employee_id                0
transaction_date           0
expense_approved_amount    0
reporting_currency         0
approval_status            0
dtype: int64

In [8]:
# AMOUNT & DELAY LOGIC

invalid_amounts = df[df["expense_approved_amount"] <= 0]
negative_delays = df[df["submission_delay_days"] < 0]
len(invalid_amounts), len(negative_delays)

(0, 0)

In [9]:
# APPROVAL STATUS CONSISTENCY
df["approval_status"].value_counts(dropna=False)

approval_status
Sent Back to Employee            17620
Approved                         17613
Submitted & Pending Approval     17500
Approved in Accounting Review    17267
Name: count, dtype: int64

In [10]:
VALID_APPROVAL_STATUS = {
    "Approved",
    "Approved in Accounting Review",
    "Submitted & Pending Approval",
    "Sent Back to Employee",
    "Rejected"
}

In [11]:
invalid_approval_status = df[
    ~df["approval_status"].isin(VALID_APPROVAL_STATUS)
]
len(invalid_approval_status)

0

In [12]:
# ACTIVE EMPLOYEE FLAG VALIDATION

df["active"].value_counts(dropna=False)

active
No     35158
Yes    34842
Name: count, dtype: int64

In [14]:
VALID_ACTIVE_FLAGS = {"Yes", "No"}
invalid_active_flag = df[~df["active"].isin(VALID_ACTIVE_FLAGS)]
len(invalid_active_flag)

0

In [15]:
# CURRENCY INTEGRITY CHECKS
# Reimbusement vs Reporting Currency

missing_currency = df[
    df["reimbursement_currency"].isnull() |
    df["reporting_currency"].isnull()
]
len(missing_currency)

0

In [16]:
fx_mismatch = df[
    (df["expense_approved_amount"] > 0)&
    (df["expense_approved_amount_rpt"].isnull())
]
len(fx_mismatch)

0

In [18]:
fx_zero_cases = df[
    df["expense_approved_amount_rpt"] <= 0
]
len(fx_zero_cases)

0

In [20]:
# Temporal Order Validation

invalid_temporal_chain = df[
    (df["first_submitted_date"] < df["transaction_date"]) |
    (df["manager_approval_date"] < df["first_submitted_date"]) |
    (df["accounting_approval_date"] < df["manager_approval_date"])
]
len(invalid_temporal_chain)

0

In [ ]:
# Submission Delay Cross-Check

computed_delay = (
    df["first_submitted_date"] - df["transaction_date"]
).dt.days

In [22]:
delay_mismatch = df[
    df["submission_delay_days"] != computed_delay
]
len(delay_mismatch)

0

In [23]:
# Attendee Logic (Policy Risk)

invalid_attendees = df[
    df["number_of_attendees"] < 0
]
len(invalid_attendees)

0

In [25]:
large_attendees =df[
    df["number_of_attendees"] > 20
]
len(large_attendees)

0

In [26]:
# Categorical Completeness 

DIMENSION_COLUMNS = [
    "department_name",
    "job_family",
    "expense_type",
    "parent_expense_type",
    "region",
    "country"
]

dim_nulls = (
    df[DIMENSION_COLUMNS]
    .isnull()
    .mean()
    .mul(100)
    .round(2)
)

dim_nulls

department_name        0.0
job_family             0.0
expense_type           0.0
parent_expense_type    0.0
region                 0.0
country                0.0
dtype: float64

In [27]:
# Duplicate Business Keys

BUSINESS_KEY = [
    "employee_id",
    "transaction_date",
    "expense_approved_amount",
    "expense_type"
]

In [28]:
business_duplicates = df[
    df.duplicated(subset=BUSINESS_KEY, keep=False)
]
len(business_duplicates)

0

In [29]:
# Outlier Flagging (NO Removal)

p99 = df["expense_approved_amount_rpt"].quantile(0.99)

df["is_high_amount_outlier"] = (
    df["expense_approved_amount_rpt"] > p99
)

In [31]:
df["approval_status"].value_counts()

approval_status
Sent Back to Employee            17620
Approved                         17613
Submitted & Pending Approval     17500
Approved in Accounting Review    17267
Name: count, dtype: int64

In [32]:
validation_results = [
    # HARD FAILS
    ("missing_columns", len(missing), "FAIL", "Schema integrity"),
    ("mandatory_nulls", df[MANDATORY_COLUMNS].isnull().any(axis=1).sum(), "FAIL", "KPI math"),
    ("invalid_amounts", len(invalid_amounts), "FAIL", "Spend KPIs"),
    ("fx_missing", len(missing_currency), "FAIL", "FX KPIs"),
    ("temporal_chain_invalid", len(invalid_temporal_chain), "FAIL", "Cycle time KPIs"),
    ("negative_submission_delay", len(negative_delays), "FAIL", "Cycle time KPIs"),

    # WARNINGS
    ("invalid_approval_status", len(invalid_approval_status), "WARN", "Approval KPIs"),
    ("invalid_active_flag", len(invalid_active_flag), "WARN", "HR slice"),
    ("delay_mismatch", len(delay_mismatch), "WARN", "Latency metrics"),
    ("large_attendees", len(large_attendees), "WARN", "Policy risk"),
    ("business_duplicates", len(business_duplicates), "WARN", "Aggregation risk"),
]

validation_df = pd.DataFrame(
    validation_results,
    columns=["check", "count", "severity", "kpi_impact"]
)

validation_df


,check,count,severity,kpi_impact
0,missing_columns,0,FAIL,Schema integrity
1,mandatory_nulls,0,FAIL,KPI math
2,invalid_amounts,0,FAIL,Spend KPIs
3,fx_missing,0,FAIL,FX KPIs
4,temporal_chain_invalid,0,FAIL,Cycle time KPIs
5,negative_submission_delay,0,FAIL,Cycle time KPIs
6,invalid_approval_status,0,WARN,Approval KPIs
7,invalid_active_flag,0,WARN,HR slice
8,delay_mismatch,0,WARN,Latency metrics
9,large_attendees,0,WARN,Policy risk


In [33]:
hard_fail_count = validation_df.query("severity == 'FAIL' and count > 0").shape[0]

if hard_fail_count > 0:
    print("❌ DATA VALIDATION FAILED — Fix issues before KPI layer")
else:
    print("✅ DATA VALIDATION PASSED — Safe to proceed to KPI layer")


✅ DATA VALIDATION PASSED — Safe to proceed to KPI layer
